# Phase 3 — context-engine comparison study

**Backfill for Day 15** (run date 2026-05-18). Reads the canonical artifacts already on disk and re-renders the comparison tables + charts in notebook form. No new experiments run here — this is the analysis surface the SKILL's project structure prescribes for Phase 3.

**Inputs read:**
* `results/phase3_context_engine_results.json` — top-level dict; `results` key holds 200 pairs × 5 strategies = 1000 rows, each with brief tokens / latency / quality / source mix.
* `results/phase3_day15_analysis.json` — per-strategy aggregates + hybrid vs recency/summarized per-pair beat/match/lose buckets.
* `benchmarks/data/manifest.json` — dataset counts + bucket distribution.

**Outputs produced:**
* Per-strategy headline table (mean tokens, mean quality, tokens/quality-unit).
* Hybrid head-to-head bar charts (vs recency, vs summarized).
* Per-bucket token compression chart (short / medium / long / very_long).
* Charts saved to `results/phase3_context_engine_analysis_*.png` for reuse in Phase-3 wrap deck.

**LLM mode of the source numbers:** mock-proxy quality scoring. The Day-18 Phase-3 wrap re-runs against real LLM judge once `LLM_PROVIDER=anthropic` is wired.

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
DATA = ROOT / "benchmarks" / "data"

results_blob = json.loads((RESULTS / "phase3_context_engine_results.json").read_text(encoding="utf-8"))
analysis = json.loads((RESULTS / "phase3_day15_analysis.json").read_text(encoding="utf-8"))
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))

# Harness writes top-level dict with metadata + 'results' list of per-(pair, strategy) rows.
rows = results_blob["results"] if isinstance(results_blob, dict) and "results" in results_blob else results_blob

print(f"rows: {len(rows)}; strategies: {sorted({r['strategy'] for r in rows})}")
print(f"buckets: {manifest['bucket_counts']}")
print(f"judge_mode: {analysis['judge_mode']}")

## 1. Headline per-strategy table

Mean brief tokens, mean quality (mock-proxy), and `tokens / quality-unit` (lower is better — fewer tokens per quality point).

In [ ]:
per_strategy = analysis["per_strategy"]
order = ["naive_dump", "recency", "semantic", "summarized", "hybrid"]

print(f"{'strategy':<12} {'mean_tokens':>13} {'mean_quality':>13} {'tokens/qual':>13}")
print("-" * 56)
for s in order:
    e = per_strategy[s]
    print(f"{s:<12} {e['mean_brief_tokens']:>13.1f} {e['mean_quality']:>13.3f} {e['tokens_to_quality_unit']:>13.1f}")

print()
print(f"proxy definition: {analysis['proxy_definition']}")
print(f"bias note: {analysis['proxy_bias_note']}")

## 2. Hybrid head-to-head (per-pair)

How often does hybrid beat, match, or lose to recency / summarized on the same pair?

In [ ]:
vs_recency = analysis["hybrid_vs_recency"]
vs_summary = analysis["hybrid_vs_summarized"]

print("hybrid vs recency (per pair):")
for k, v in vs_recency.items():
    print(f"  {k:<10} {v}")
print("\nhybrid vs summarized (per pair):")
for k, v in vs_summary.items():
    print(f"  {k:<10} {v}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, data in [
    (axes[0], "hybrid vs recency", vs_recency),
    (axes[1], "hybrid vs summarized", vs_summary),
]:
    labels = [k for k in ("beat", "match", "lose") if k in data]
    values = [data[k] for k in labels]
    colors = {"beat": "#2ca02c", "match": "#7f7f7f", "lose": "#d62728"}
    ax.bar(labels, values, color=[colors[l] for l in labels])
    ax.set_title(title)
    ax.set_ylabel("pairs")
    for i, v in enumerate(values):
        ax.text(i, v + 2, str(v), ha="center")
fig.tight_layout()
out = RESULTS / "phase3_context_engine_analysis_head_to_head.png"
fig.savefig(out, dpi=120)
print(f"saved {out}")

## 3. Per-bucket brief tokens (where hybrid actually compresses)

The hybrid story is "compress only when there's something to compress" — short / medium histories fit under 8K so hybrid and recency are essentially identical; long / very_long histories are where hybrid's cold-tail summary kicks in.

In [ ]:
by_bucket: dict[str, dict[str, list[float]]] = defaultdict(lambda: defaultdict(list))
for r in rows:
    # Row carries 'history_bucket' OR is joinable to the pair on 'pair_id'.
    bucket = r.get("history_bucket")
    if bucket is None:
        # Lazy-join via the pair_id → manifest bucket map if needed.
        continue
    by_bucket[bucket][r["strategy"]].append(r["brief_tokens_estimate"])

if not by_bucket:
    # Fallback: rows don't carry history_bucket — join via the dataset loader.
    from benchmarks.dataset_loader import load_pairs  # type: ignore
    bucket_for = {p.pair_id: p.history_bucket for p in load_pairs()}
    for r in rows:
        b = bucket_for.get(r["pair_id"])
        if b is not None:
            by_bucket[b][r["strategy"]].append(r["brief_tokens_estimate"])

buckets = ["short", "medium", "long", "very_long"]
labels = ["recency", "semantic", "summarized", "hybrid", "naive_dump"]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(buckets))
width = 0.15
for i, strat in enumerate(labels):
    means = [
        (sum(by_bucket[b][strat]) / len(by_bucket[b][strat])) if by_bucket[b][strat] else 0
        for b in buckets
    ]
    ax.bar(x + (i - 2) * width, means, width, label=strat)
ax.set_xticks(x)
ax.set_xticklabels(buckets)
ax.set_ylabel("mean brief tokens")
ax.set_title("Per-bucket mean brief tokens (lower is cheaper)")
ax.legend()
fig.tight_layout()
out = RESULTS / "phase3_context_engine_analysis_per_bucket_tokens.png"
fig.savefig(out, dpi=120)
print(f"saved {out}")

## 4. Findings

Pulled from `results/phase3_day15_analysis.json` + `results/EXPERIMENT_LOG.md`:

1. **Hybrid is the cost-frontier strategy under mock-mode quality.** Emits 41% smaller briefs than recency on aggregate while matching recency's fact recall on 183 of 200 pairs (91.5%). The 17 losses are all on long / very_long buckets and the loss is the mock-summary's fault, not the strategy's.
2. **Naive / recency / semantic emit byte-identical briefs on 95% of pairs** under the 8K token budget. They score identically under the mock proxy because most histories fit verbatim. The semantic-vs-recency comparison is only meaningful on the 10 very_long pairs, and even there both strategies preserve the same fact-token set.
3. **Token compression at parity quality:** hybrid 1,338 tokens (long) vs recency 3,014 (56% smaller); hybrid 1,274 (very_long) vs recency 7,887 (84% smaller).
4. **Bias note:** mock proxy under-states what a real LLM judge would credit a coherent summary with. Day-18 re-judge with `LLM_PROVIDER=anthropic` reads the same `phase3_context_engine_results.json` artifact unchanged and re-fills the `quality_score` column with real-LLM numbers.

## Next analysis to land here

* **Day-17 / Day-18** — pair this notebook with `phase3_orchestrator_analysis.ipynb` (Day 18) for the consolidated Phase-3 wrap deck.
* **Day-27 (Phase 5)** — re-run with real-LLM quality column and add the naive-baseline comparison frame the SKILL prescribes.